# **Structure Unstructured Restaurant Data with an LLM**

## **The challenge**

We have a large, messy dataset spanning multiple modalities:

* **Restaurant descriptions** and **user reviews** (raw TXT)
* **Food recipes** with structured metadata and images (JSON + JPEG)
* **User restaurant visit histories** (JSON with URLs)

However, this data is not immediately usable:

1. **Unstructured text** lacks a consistent schema, making search and retrieval inefficient.
2. **Images and URLs** are opaque to traditional data pipelines and must be transformed into searchable, quantitative representations.
3. The data spans multiple modalities with no unified representation.

To build a high-quality, explainable recommendation engine, we need to transform this **heterogeneous, multimodal data** into a **structured knowledge base**. This includes extracting semantic signals from **text**, generating representations for **images** and **URLs**, and organizing everything into a coherent, query-friendly JSON format.


## **Steps**


Our Python program will:

* Load raw restaurant descriptions and review data from unstructured TXT files
* Use a multimodal/LLM-based pipeline to extract structured attributes (e.g., cuisine type, ambiance, dietary options, and signature dishes)
* Convert the extracted information into a well-defined JSON schema suitable for indexing and search


----


## **Set up the lab environment**


For this lab, we will be using the following libraries:

* [`json`](https://docs.python.org/3/library/json.html) for parsing, constructing, and serializing structured JSON representations extracted from unstructured text
* [`python-dotenv`](https://pypi.org/project/python-dotenv/) for loading environment variables from .env file
* [`pydantic`](https://pydantic.dev/docs/validation/latest/get-started/) for object blueprint creation and validation

* **IBM Watsonx AI SDK** (`ibm-watsonx-ai`) for interacting with foundation models hosted on IBM watsonx.ai:

  * `Credentials` to securely authenticate with the watsonx.ai service
  * `ModelInference` to invoke foundation models for text understanding and information extraction


### Import required libraries

It is recommended to import all required libraries in one place (here):


In [1]:
import json
import os
from dotenv import load_dotenv
import time

# IBM WatsonX imports
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

# Validation
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

from config import config

# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')


## **Step 1: Load the data and display the texts**

### a. load the text data file and explore the restaurant contents.

In [2]:
file_path = "California-Culinary-Map.txt"

# Open the text file and read its contents
with open(file_path, 'r') as file:
    data = file.read()

# Print the first 100 characters of the restaurant data
print(data[:1000])

### The Culinary Map of California

**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$

Down in **Santa Monica**, **Mar de Cortez** serves as a **sun-drenched**, **casual taqueria** specializing in **Baja-style seafood**. With a **4.2/5** rating, it captures the salt-air energy of the coast through its signature beer-battered snapper tacos and zesty octopus ceviche, making it a premier spot for open-air dining near the pier. Price range: $

**Iron & Embers** anchors the **Arts District** in **DTLA** as a **moody**, **industrial American steakhouse**. This **fine dining** establishment earns a **4.8/5** for its com

### b. Split the restaurant paragraphs into a Python list

In the previous step, we noticed that the text file consists of multiple paragraphs separated by two newline characters, each describing a single restaurant. Let's split these paragraphs into a list, allowing us to work with and manage each restaurant’s data more effectively.


In [67]:
# Split the restaurant paragraphs into list
restaurant_list = data.split('\n\n')

# Remove the first item since it is the dataset name
restaurant_list = restaurant_list[1:]

# Print the number of restaurants and the first restaurant description
print(f"Number of restaurants: {len(restaurant_list)}\n")
print(f"First restaurant description: {restaurant_list[0]}")

Number of restaurants: 210

First restaurant description: **The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$


----


## **Step 2: Define the LLM**

Key observation: each restaurant description includes the following key attributes: 
1. Restaurant name
2. Location
3. Restaurant type
4. Food style
5. Rating
6. Price range
7. Signature dishes
8. Specialties
9. Shortcomings

This information is essential for uniquely identifying and describing a restaurant. We will use a large language model (LLM) to organize these attributes into a structured JSON format.


### a. Implement the LLM function

We will use **IBM Granite 4H Small** as the base LLM for this lab, as it provides strong text understanding capabilities while remaining cost-efficient

In [3]:
def llm_model(system_msg, prompt_txt):
    """
    Function to interact with the IBM WatsonX AI LLM model.

    Args:
        system_msg (str): The system message to set the context for the LLM.
        prompt_txt (str): The user prompt to be sent to the LLM.
    
    Returns:
        str: The output text generated by the LLM in response to the prompt.
    """
    
    model_id = "ibm/granite-4-h-small"

    credentials = Credentials(
        url = config["WATSONX_URL"],
        api_key = config["WATSONX_API_KEY"]
    )

    # Define the model by ModelInference
    model = ModelInference(
        model_id = model_id,
        credentials = credentials,
        project_id = config["WATSONX_PROJECT_ID"],
    )

    # Define the messages
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_txt}
    ]

    # 1.3: Get the final response output and return it
    response = model.chat(messages=messages)
    output_text = response["choices"][0]["message"]["content"]
    
    return output_text

### b. Test the llm_model function

In [ ]:
def safe_llm_call(system_msg, prompt_txt, retries=3):
    for i in range(retries):
        try:
            return llm_model(system_msg, prompt_txt)
        except Exception:
            time.sleep(2)
    return f"Failed after {retries} retries"

In [82]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(safe_llm_call(system_msg, prompt_txt))

Hawaii is much warmer than Greenland in winter. Here's a comparison:

1. Hawaii:
   - Located in the Pacific Ocean
   - Winter temperatures typically range from the mid-60s to mid-70s °F (around 18-24 °C)
   - Mild and relatively consistent temperatures due to its tropical location

2. Greenland:
   - An autonomous country within the Kingdom of Denmark, located in the Arctic region of North America
   - Winter temperatures can drop well below freezing, often ranging from -20°F to 20°F (-29°C to -6°C)
   - Cold temperatures due to its high latitude and proximity to the Arctic

In winter, even the warmest parts of Greenland are much colder than the coolest parts of Hawaii. For example, in Kangerlussuaq, Greenland (one of the warmest places in Greenland), average winter temperatures range from -4°F to 9°F (-20°C to -13°C). Meanwhile, in Honolulu, Hawaii (one of the warmest parts of Hawaii), winter temperatures typically range from 65°F to 78°F (18°C to 26°C).

Therefore, Hawaii experience

----


## **Step 3: Prompt Engineering**

To make an LLM behave reliably and produce outputs that meet our requirements, careful prompt design is essential. We will define a prompt template that follows the best below key prompt engineering techniques:

* **Clear and specific instructions**: Explicitly state the task, expected output format, and constraints to reduce ambiguity.
* **Structured output guidance**: Provide schemas, templates, or examples (e.g., JSON formats) to encourage consistent, machine-readable responses.
* **Few-shot prompting**: Include representative input–output examples to guide the model toward the desired behavior.
* **Role prompting**: Assign the model a specific role (e.g., “You are a data extraction assistant”) to shape its reasoning and tone.
* **Constraint-based prompting**: Define what the model should and should not do to improve precision and reliability.
* **Iterative refinement**: Evaluate outputs and progressively refine prompts to improve performance over time.

### a. Define the template

Here, we will implement a function that generates prompts for the LLM, given an input restaurant description paragraph.
We will apply the **one-shot prompting** technique, which includes a single example of the desired output within the prompt to guide the model’s response.
An example output is provided, corresponding to the second restaurant paragraph in our `restaurant_list` variable. Using this example, we will design a prompt template that enables the model to consistently transform *any* input restaurant description into the required structured JSON format.

**Note:** For the price range field, we will instruct the LLM to convert dollar signs (e.g., \\\\\$, \\$\\$, $$$) into an integer representing the number of dollar symbols.


In [83]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1] #use the second restaurant paragraph as the example
EXAMPLE_OUTPUT = """
    {
    "name": "Mar de Cortez",
    "location": "Santa Monica",
    "type": "casual taqueria",
    "food_style": "Baja-style seafood",
    "rating": 4.2,
    "price_range": 1,
    "signatures": [
        "beer-battered snapper tacos",
        "zesty octopus ceviche"
    ],
    "vibe": "salt-air energy",
    "environment": "a premier sun-drenched spot for open-air dining near the pier.",
    "shortcomings": []
    }
"""

### Design your prompt here
def restaurant_data_structure_prompt_generation(restaurant_paragraph):
    base_system_msg = f"""
    You are a precise data extraction assistant specialized in restaurant information.
    Extract structured data exactly as specified, following the output format strictly.
    """
    
    base_user_prompt = f"""
    Task: Extract restaurant information from the description below into a valid JSON object with these exact fields:
    Required fields:
    - name (string): The restaurant's name
    - location (string): The city or area where it's located  
    - type (string): The restaurant category (e.g., "casual taqueria", "fine dining")
    - food_style (string): The cuisine style (e.g., "Baja-style seafood", "Italian")
    - rating (float): Numerical rating from 0-5, extract from patterns like "4.2/5" or "4.2 out of 5"
    - price_range (int): Count the number of "$" symbols at the end of the description (1-4)
    - signatures (string[]): List of signature dishes mentioned
    - vibe (string): The atmosphere or feeling (e.g., "salt-air energy", "cozy and romantic")
    - environment (string): A complete sentence describing the setting
    - shortcomings (string[]): List any negative aspects mentioned, or empty array if none
    
    Extraction rules:
    1. When multiple adjectives describe the restaurant, combine them into the appropriate field
    2. If a rating is formatted as "X/Y", extract only the X value
    3. Count only consecutive "$" symbols at the very end of the text
    4. Signature dishes are often preceded by words like "signature", "specializing in", or "known for"
    5. The vibe should capture the emotional or atmospheric quality

    Restaurant description:
    {restaurant_paragraph}

    Example:
    Input Restaurant Description: {EXAMPLE_RESTAURANT_PARAGRAPH}
    Output:
    {EXAMPLE_OUTPUT}

    Important: Return ONLY the JSON object, no additional text or explanations.
    
    """
    return base_system_msg, base_user_prompt

### b. Let's test the prompts

In [84]:
# Unit test:
restaurant_paragraph = restaurant_list[10]
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph)

test_response = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)
print(test_response)

{
    "name": "The Emerald Garden",
    "location": "Culver City",
    "type": "lush Vietnamese bistro",
    "food_style": "Vietnamese",
    "rating": 4.6,
    "price_range": 4,
    "signatures": [],
    "vibe": "tropical escape hidden in the city",
    "environment": "a refreshing, breezy environment for a mid-week dinner.",
    "shortcomings": []
}


### c. Validate the LLM outputs

LLMs are not always perfectly reliable and may produce errors, even when strict rules are specified in the prompt. To ensure that the generated outputs strictly conform to the required JSON format, we will define a function that formally validates the LLM output. 


In [ ]:
# Define the schema
class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)


# Use the validation method to validate the test_response from the unit test
try:
    restaurant_data = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data.name}")
except ValidationError as e:
    print(f"Validation failed: {e.json()}")

Success! Validated: The Emerald Garden


----


## **Step 4: Structure all the restaurant data**
### a. Define prompts to have an LLM that auto repairs outputs to JSON format

As mentioned earlier, LLMs are not perfect. While we previously defined a schema to validate the output, validation alone does not fix incorrect results. To address this, we will introduce an additional LLM “expert” whose sole responsibility is to automatically repair and correct outputs that do not conform to the required JSON format.

In this step, we will define the prompt template function for this LLM. This function takes:
- `candidate_json_output`: The candidate json output
- `error_message`: The error message generated by the schema validation if a mistake is detected
- `original_paragraph`: The original restaurant description to validate data accuracy

In our prompts, we will tell LLM:
- The original wrong output by feeding `candidate_json_output`
- The guidance on correction by feeding `error_message`
- The source of truth by feeding `original_paragraph`


In [87]:
def JSON_auto_repair_prompts(candidate_json_output: str | dict, error_message: str, original_paragraph: str =None):
    """
    Generate prompts to repair malformed JSON or invalid data.
    
    Args:
        candidate_json_output (str | dict): The invalid JSON string or data
        error_message (str): The error message from validation
        original_paragraph (str): Optional original restaurant description for context
    Returns:
        Tuple(str, str) 
    """
    auto_repair_system_msg = """
    You are an expert JSON repair assistant specializing in fixing restaurant data schema violations. 
    Your goal is to correct JSON data to match the Restaurant schema while preserving as much original information as possible.
    You must:
    1. Fix malformed JSON syntax
    2. Correct data types to match schema expectations  
    3. Maintain all valid data while only fixing what's broken
    4. Return ONLY the repaired JSON object, no additional text
    """
    
    context_prompt = ""
    if original_paragraph:
        context_prompt = f"""
        Original restaurant description for context:
        {original_paragraph}
        
        Use this to infer missing or incorrect values (e.g., if a price_range is missing, count $ symbols in this text).
        """
    auto_repair_prompt = f"""
    I need to repair this restaurant data JSON to match our validation schema.

    Current (invalid) JSON data:
    {json.dumps(candidate_json_output, indent=2) if isinstance(candidate_json_output, dict) else candidate_json_output}
    
    Validation Error Details:
    {error_message}

    Schema Requirements:
    - name: string (required)
    - location: string (required) 
    - type: string (required)
    - food_style: string (required)
    - rating: float, optional (must be 0-5)
    - price_range: int, optional
    - signatures: array of strings (default: empty array)
    - vibe: string, optional
    - environment: string (required)
    - shortcomings: array of strings (default: empty array)
    
    Repair Instructions:
    1. Fix any JSON syntax errors
    2. Ensure all required fields are present
    3. Correct data types (e.g., "4.2" should be 4.2, "1" should be 1)
    4. If values are missing but can be inferred from the context, add them
    5. Keep all valid data intact
    6. Return ONLY the repaired JSON, no explanations or markdown

    {context_prompt}
    Apply these specific fixes:
    - For rating: ensure it's a float between 0-5, not a string
    - For price_range: ensure it's an integer, not a string or $ symbol
    - For arrays (signatures, shortcomings): ensure they're arrays, not strings
    - For required fields: if missing, use reasonable defaults or infer from context
    """
    return auto_repair_system_msg, auto_repair_prompt

### b. Run the for loop to go over all the restaurant data in the list (approximately 20 minutes)

We will structure each restaurant description paragraph into a JSON-formatted output by iterating through the dataset using a for loop.
Although using an LLM to auto repair the response format is not the best practice, here we are guaranteed that the generation and repair tasks here are simple enough for LLMs to accomplish.

In [88]:
structured_restaurant_lists = []
for i, restaurant_paragraph in enumerate(restaurant_list):
    # Produce initial output
    base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph)
    candidate_json_output = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)

    # Validation and Auto Correction loop on the output
    is_valid = False
    
    while not is_valid:
        try:
            restaurant_data = Restaurant.model_validate_json(candidate_json_output)
            is_valid = True
        except ValidationError as e:
            auto_repair_system_msg, auto_repair_prompt = JSON_auto_repair_prompts(candidate_json_output, str(e))
            candidate_json_output = llm_model(system_msg=auto_repair_system_msg, prompt_txt=auto_repair_prompt)

    # 2.3: Append the finalized response to the structured_restaurant_lists
    structured_restaurant_lists.append(candidate_json_output)

    # A manual progress bar
    if (i+1)%20 == 0:
        print(f'{i+1} out of {len(restaurant_list)} is done')

# A final message to notify the completion
print('ALL DONE!!')

20 out of 210 is done
40 out of 210 is done
60 out of 210 is done
80 out of 210 is done
100 out of 210 is done
120 out of 210 is done
140 out of 210 is done
160 out of 210 is done
180 out of 210 is done
200 out of 210 is done
ALL DONE!!


### c. Save the list to a JSON file

In [89]:
# Print the 50th item in the structured_restaurant_lists
print(structured_restaurant_lists[49])

{
    "name": "The Neon Noodle Bar",
    "location": "Monterey Park",
    "type": "fast and funky Hong Kong cafe",
    "food_style": "Hong Kong",
    "rating": 4.0,
    "price_range": 2,
    "signatures": [],
    "vibe": "frenetic and delicious energy of a late-night HK diner",
    "environment": "a high-energy spot for instant noodle gourmet bowls and pineapple buns with cold butter.",
    "shortcomings": []
}


In [90]:
structured_restaurant_lists_json = []

for elt in structured_restaurant_lists:
    try:
        structured_restaurant_lists_json.append(json.loads(elt))
    except Exception as e:
        print(e)

#For each item in the restaurant list, assign it with an itemId to be consistent with the one in the user review data:
for i, response in enumerate(structured_restaurant_lists_json):
    response['itemId'] = 1000001 + i
    structured_restaurant_lists_json[i] = response
    
filename = 'structured_restaurant_data.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(structured_restaurant_lists_json, f, indent=4)

## **Conclusion**

We've successfully transformed raw restaurant data into a wel structured, JSON-formatted data.